# Exercise 3: Cybersecurity Incident Report

Student Name: Segundo
Student ID: TUPM-26-1952

Target Year: 2017

In [3]:
from pathlib import Path
import csv
import json
import shutil

LAST_NAME = "Segundo"
STUDENT_ID = "TUPM-26-1952"

SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_YEAR = 2015 + SEED_DIGIT

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Target Year:", TARGET_YEAR)

Student: Segundo
Student ID: TUPM-26-1952
Seed Digit: 2
Target Year: 2017


In [4]:
BASE_DIR = Path.cwd()

if not (BASE_DIR / "Global_Cybersecurity_Threats_2015-2024.csv").exists() and (BASE_DIR / "Additional_Exercise_3").exists():
    BASE_DIR = BASE_DIR / "Additional_Exercise_3"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

with open(
    BASE_DIR / "Global_Cybersecurity_Threats_2015-2024.csv",
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:

    reader = csv.DictReader(file)
    records = list(reader)

print("Total records:", len(records))
print("Available columns:")

for column in records[0].keys():
    print(column)

Total records: 3000
Available columns:
Country
Year
Attack Type
Target Industry
Financial Loss (in Million $)
Number of Affected Users
Attack Source
Security Vulnerability Type
Defense Mechanism Used
Incident Resolution Time (in Hours)


In [5]:
target_records = []

for row in records:
    try:
        year = int(float(row["Year"]))

        if year == TARGET_YEAR:
            target_records.append(row)

    except (ValueError, TypeError):
        continue

print("Total records:", len(records))
print("Incidents in target year:", len(target_records))

Total records: 3000
Incidents in target year: 319


In [6]:
from collections import Counter

attack_counts = Counter(
    row["Attack Type"].strip()
    for row in target_records
    if row["Attack Type"].strip()
)

print("Incident count per attack type:")

for attack, count in sorted(attack_counts.items()):
    print(attack, ":", count)

Incident count per attack type:
DDoS : 58
Malware : 46
Man-in-the-Middle : 58
Phishing : 64
Ransomware : 44
SQL Injection : 49


In [7]:
def parse_loss(value):
    try:
        cleaned = value.replace(",", "").replace("$", "").strip()
        return float(cleaned)
    except (ValueError, TypeError, AttributeError):
        return None

valid_losses = []

for row in target_records:
    loss = parse_loss(row["Financial Loss (in Million $)"])

    if loss is not None:
        valid_losses.append(loss)

total_loss = sum(valid_losses)

average_loss = (
    total_loss / len(valid_losses)
    if valid_losses else 0
)

print("Total Financial Loss:", round(total_loss, 2), "million USD")
print("Average Financial Loss:", round(average_loss, 2), "million USD")
print("Valid Financial Records:", len(valid_losses))

Total Financial Loss: 16261.68 million USD
Average Financial Loss: 50.98 million USD
Valid Financial Records: 319


In [8]:
ranked_incidents = []

for row in target_records:
    loss = parse_loss(row["Financial Loss (in Million $)"])

    if loss is not None:
        ranked_incidents.append((row, loss))

ranked_incidents.sort(
    key=lambda item: item[1],
    reverse=True
)

top_five = ranked_incidents[:5]

print("FIVE COSTLIEST INCIDENTS")

for rank, (row, loss) in enumerate(top_five, start=1):
    print("\nRank:", rank)
    print("Year:", row["Year"])
    print("Attack Type:", row["Attack Type"])
    print("Financial Loss:", loss, "million USD")

FIVE COSTLIEST INCIDENTS

Rank: 1
Year: 2017
Attack Type: SQL Injection
Financial Loss: 99.99 million USD

Rank: 2
Year: 2017
Attack Type: Malware
Financial Loss: 99.22 million USD

Rank: 3
Year: 2017
Attack Type: SQL Injection
Financial Loss: 98.75 million USD

Rank: 4
Year: 2017
Attack Type: DDoS
Financial Loss: 98.73 million USD

Rank: 5
Year: 2017
Attack Type: DDoS
Financial Loss: 98.61 million USD


In [9]:
filtered_csv = OUTPUT_DIR / "filtered_cybersecurity_incidents.csv"

with open(filtered_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=records[0].keys()
    )

    writer.writeheader()
    writer.writerows(target_records)

print("Filtered CSV created:", filtered_csv)

Filtered CSV created: c:\Users\jerma\Lab6_File_Handling_Additional_Exercises\Additional_Excercise_3\data\output\filtered_cybersecurity_incidents.csv


In [10]:
top_five_data = []

for rank, (row, loss) in enumerate(top_five, start=1):
    top_five_data.append({
        "rank": rank,
        "year": row["Year"],
        "attack_type": row["Attack Type"],
        "financial_loss_millions": loss
    })

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": len(records),
    "incidents_in_target_year": len(target_records),
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_millions": round(total_loss, 2),
    "average_financial_loss_millions": round(average_loss, 2),
    "valid_financial_records": len(valid_losses),
    "five_costliest_incidents": top_five_data,
    "filtered_csv": str(filtered_csv)
}

json_file = OUTPUT_DIR / "cybersecurity_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_file)

JSON summary created: c:\Users\jerma\Lab6_File_Handling_Additional_Exercises\Additional_Excercise_3\data\output\cybersecurity_summary.json


In [11]:
backup_file = BACKUP_DIR / "cybersecurity_summary_backup.json"

shutil.copy2(json_file, backup_file)

print("Backup file:", backup_file)
print("Backup exists:", backup_file.exists())

with open(backup_file, "r", encoding="utf-8") as file:
    backup_data = json.load(file)

print("Backup verification:",
      backup_data == summary)

Backup file: c:\Users\jerma\Lab6_File_Handling_Additional_Exercises\Additional_Excercise_3\data\backup\cybersecurity_summary_backup.json
Backup exists: True
Backup verification: True


In [12]:
report_file = OUTPUT_DIR / "cybersecurity_report.txt"

with open(report_file, "w", encoding="utf-8") as file:

    file.write("CYBERSECURITY INCIDENT REPORT\n")
    file.write("=" * 40 + "\n\n")

    file.write(f"Student: {LAST_NAME}\n")
    file.write(f"Student ID: {STUDENT_ID}\n")
    file.write(f"Target Year: {TARGET_YEAR}\n\n")

    file.write(f"Total Dataset Records: {len(records)}\n")
    file.write(f"Incidents in Target Year: {len(target_records)}\n\n")

    file.write("INCIDENT COUNTS PER ATTACK TYPE\n")

    for attack, count in sorted(attack_counts.items()):
        file.write(f"{attack}: {count}\n")

    file.write("\nFINANCIAL SUMMARY\n")
    file.write(f"Total Loss: {total_loss:.2f} million USD\n")
    file.write(f"Average Loss: {average_loss:.2f} million USD\n")

    file.write("\nFIVE COSTLIEST INCIDENTS\n")

    for rank, (row, loss) in enumerate(top_five, start=1):
        file.write(
            f"{rank}. {row['Attack Type']} - "
            f"{loss:.2f} million USD\n"
        )

    file.write("\nOUTPUT FILES\n")
    file.write(f"Filtered CSV: {filtered_csv}\n")
    file.write(f"JSON Summary: {json_file}\n")
    file.write(f"JSON Backup: {backup_file}\n")
    file.write(f"Report: {report_file}\n")

print("Report created:", report_file)

Report created: c:\Users\jerma\Lab6_File_Handling_Additional_Exercises\Additional_Excercise_3\data\output\cybersecurity_report.txt
